# affect_aif Demo Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/har5h1l/affect_aif/blob/master/notebooks/demo.ipynb)

Run small, Colab-compatible trust-task experiments and inspect the generated outputs. This notebook is intentionally demo-scale: it exercises the same public runner and analysis path as the paper workflow, but with fewer seeds and shorter episodes.

Default behavior writes scratch outputs under `outputs/notebook_demo/`, not canonical `results/`, so your clean result packet remains shareable.


## What This Notebook Proves

This is a run-first notebook, not a precomputed-results viewer. Each experiment section has one execution cell that runs the TOML config through `scripts/experiment/run.py` and then runs `scripts/analysis/analyze.py` on the fresh `results.csv`. The following cells in that section load, plot, and summarize only that experiment.

Use it to understand the workflow before launching the full paper reproduction notebook.


## 1. Bootstrap Runtime

This cell works in two modes:

- **Google Colab:** clones the repo branch and moves into it.
- **Local Jupyter:** assumes you launched the notebook from the repo root.

Use a GPU runtime if available, but the demo also runs on CPU.


In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap

IN_COLAB = Path("/content").exists()
REPO_URL = os.environ.get("AFFECT_AIF_REPO_URL", "https://github.com/har5h1l/affect_aif.git")
BRANCH = os.environ.get("AFFECT_AIF_BRANCH", "master")

if IN_COLAB:
    os.chdir("/content")
    if not Path("affect_aif").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, "affect_aif"], check=True)
    os.chdir("/content/affect_aif")

ROOT = Path.cwd()
if not (ROOT / "scripts" / "experiment" / "run.py").exists():
    raise RuntimeError("Run this notebook from the affect_aif repo root, or use the Colab bootstrap clone.")

print("Repo root:", ROOT)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())


## 2. Install Dependencies

Colab runtimes are fresh virtual machines, so install the repo into the runtime. Local users can set `INSTALL_DEPS = False` if the environment is already prepared.


In [ ]:
INSTALL_DEPS = True

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
    print("Installed affect_aif in editable mode for this runtime.")
else:
    print("Skipped dependency install.")


## 3. Check Runtime Device

This reports whether JAX sees CPU/GPU devices. The trust-task demo does not require a GPU, but the check makes the runtime transparent.


In [ ]:
try:
    import jax
    print("JAX devices:", jax.devices())
except Exception as exc:
    print("JAX device check unavailable:", exc)

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No NVIDIA GPU visible. This notebook also runs on CPU; GPU is optional for these trust-task sweeps.")


## 4. Demo Controls And Helpers

The helper below keeps each experiment block small. `run_demo_experiment(...)` runs exactly one config into a named scratch batch, finds the generated `results.csv`, and runs post-hoc analysis beside it.


In [ ]:
RUN_DEMO = True
RUN_ANALYSIS = True
RESET_OUTPUTS = False
OUTPUT_ROOT = Path("outputs")
DEMO_BATCH_PREFIX = "notebook_demo"

DEMO_EXPERIMENTS = {
    "model_fitness": {
        "config": "configs/demo/model_fitness.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_model_fitness",
        "experiment_id": "model_fitness_demo",
    },
    "betrayal_adaptation": {
        "config": "configs/demo/betrayal_adaptation.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_betrayal_adaptation",
        "experiment_id": "betrayal_adaptation_demo",
    },
    "alpha_sweep": {
        "config": "configs/demo/alpha_sweep.toml",
        "batch": f"{DEMO_BATCH_PREFIX}_alpha_sweep",
        "experiment_id": "alpha_sweep_demo",
    },
}

for spec in DEMO_EXPERIMENTS.values():
    if not Path(spec["config"]).exists():
        raise FileNotFoundError(spec["config"])

if RESET_OUTPUTS:
    for spec in DEMO_EXPERIMENTS.values():
        batch_dir = OUTPUT_ROOT / spec["batch"]
        if batch_dir.exists():
            shutil.rmtree(batch_dir)

import pandas as pd
import matplotlib.pyplot as plt


def run_required(cmd, required_path: Path | None = None):
    if required_path is not None and not required_path.exists():
        raise FileNotFoundError(required_path)
    print("Running:", " ".join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], check=True)


def result_paths_for_batch(batch_name: str):
    batch_dir = OUTPUT_ROOT / batch_name
    paths = sorted(batch_dir.glob("**/results.csv"))
    if not paths:
        raise RuntimeError(f"No results.csv files found under {batch_dir}. Run the experiment first.")
    return paths


def run_demo_experiment(name: str):
    spec = DEMO_EXPERIMENTS[name]
    cmd = [
        sys.executable,
        "scripts/experiment/run.py",
        "--config",
        spec["config"],
        "--output-dir",
        str(OUTPUT_ROOT),
        "--batch-name",
        spec["batch"],
        "--workers",
        "1",
    ]
    if RUN_DEMO:
        run_required(cmd)
    else:
        print("Experiment run skipped. Set RUN_DEMO = True to execute", name)

    paths = result_paths_for_batch(spec["batch"])
    if RUN_ANALYSIS:
        for result_path in paths:
            run_required(
                [
                    sys.executable,
                    "scripts/analysis/analyze.py",
                    "--results",
                    result_path,
                    "--output-dir",
                    result_path.parent / "notebook_analysis",
                ],
                required_path=result_path,
            )
    else:
        print("Analysis skipped. Set RUN_ANALYSIS = True to analyze", name)
    return paths


def load_results(paths):
    frames = []
    for path in paths:
        frame = pd.read_csv(path, low_memory=False)
        frame["source_file"] = str(path)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def summarize_by_variant(frame):
    metrics = {"payoff": ["mean", "sum"]}
    if "q_pi_entropy" in frame.columns:
        metrics["q_pi_entropy"] = ["mean"]
    return frame.groupby(["experiment_id", "variant_id"], as_index=False).agg(metrics)


def bar_metric(table, metric, title, *, by="variant_id", figsize=(8, 3.5)):
    if table.empty or metric not in table.columns or by not in table.columns:
        print(f"Cannot plot {title}: missing {metric} or {by}.")
        return
    plot = table.groupby(by, as_index=False)[metric].mean()
    fig, ax = plt.subplots(figsize=figsize)
    ax.bar(plot[by].astype(str), plot[metric])
    ax.set(title=title, xlabel=by, ylabel=metric)
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    plt.show()


## 5. Model-Fitness Demo: Run And Analyze

This short run checks the model-fitness path used by H1. It runs the demo config, writes a fresh scratch `results.csv`, and runs generic analysis on that file.


In [ ]:
model_fitness_paths = run_demo_experiment("model_fitness")
model_fitness = load_results(model_fitness_paths)
print("Rows:", len(model_fitness))
display(summarize_by_variant(model_fitness))


### Model-Fitness Demo Readout

This plots payoff and policy entropy at demo scale. Treat it as an execution sanity check, not manuscript evidence.


In [ ]:
if model_fitness.empty:
    print("No model_fitness_demo rows found.")
else:
    mf = model_fitness.groupby("variant_id", as_index=False).agg(
        mean_payoff=("payoff", "mean"),
        total_payoff=("payoff", "sum"),
        mean_entropy=("q_pi_entropy", "mean") if "q_pi_entropy" in model_fitness.columns else ("payoff", "mean"),
    )
    display(mf)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].bar(mf["variant_id"], mf["mean_payoff"])
    axes[0].set(title="Mean payoff", xlabel="Variant", ylabel="Payoff")
    axes[0].tick_params(axis="x", rotation=30)
    axes[1].bar(mf["variant_id"], mf["mean_entropy"])
    axes[1].set(title="Mean policy entropy", xlabel="Variant", ylabel="Entropy")
    axes[1].tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()
    low_entropy = mf.sort_values("mean_entropy").iloc[0]
    high_payoff = mf.sort_values("mean_payoff", ascending=False).iloc[0]
    print(f"Demo readout: lowest entropy = {low_entropy.variant_id}; highest mean payoff = {high_payoff.variant_id}.")


## 6. Betrayal-Adaptation Demo: Run And Analyze

This block runs the scheduled stance-switch demo and analyzes the generated trajectory file. The switch occurs at round 16.


In [ ]:
betrayal_paths = run_demo_experiment("betrayal_adaptation")
betrayal = load_results(betrayal_paths)
print("Rows:", len(betrayal))
display(summarize_by_variant(betrayal))


### Betrayal-Adaptation Demo Readout

The plot shows how policy entropy evolves before and after the scheduled stance switch.


In [ ]:
if betrayal.empty or "round" not in betrayal.columns:
    print("No betrayal_adaptation_demo rows found.")
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    metric = "q_pi_entropy" if "q_pi_entropy" in betrayal.columns else "payoff"
    for variant, frame in betrayal.groupby("variant_id"):
        by_round = frame.groupby("round", as_index=False)[metric].mean()
        ax.plot(by_round["round"], by_round[metric], label=variant)
    ax.axvline(16, color="black", linestyle="--", linewidth=1, label="stance switch")
    ax.set(title=f"Betrayal demo: {metric} by round", xlabel="Round", ylabel=metric)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

    window = betrayal.assign(phase=lambda d: pd.cut(d["round"], bins=[-1, 15, 50], labels=["pre", "post"]))
    phase_summary = window.groupby(["variant_id", "phase"], observed=True)[metric].mean().reset_index()
    display(phase_summary)
    print("Demo readout: compare pre/post rows to see which variants sharpen or soften policy deployment after betrayal.")


## 7. Alpha-Sweep Demo: Run And Analyze

Alpha controls affective precision gain. This block runs the short alpha sweep, analyzes the generated trajectory file, and then plots demo-scale payoff and entropy.


In [ ]:
alpha_paths = run_demo_experiment("alpha_sweep")
alpha = load_results(alpha_paths)
print("Rows:", len(alpha))
display(summarize_by_variant(alpha))


### Alpha-Sweep Demo Readout

The demo checks whether changing gain visibly changes entropy or payoff in a shortened setting.


In [ ]:
if alpha.empty:
    print("No alpha_sweep_demo rows found.")
else:
    alpha_summary = alpha.groupby("variant_id", as_index=False).agg(
        mean_payoff=("payoff", "mean"),
        total_payoff=("payoff", "sum"),
        mean_entropy=("q_pi_entropy", "mean") if "q_pi_entropy" in alpha.columns else ("payoff", "mean"),
    )
    display(alpha_summary)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].bar(alpha_summary["variant_id"], alpha_summary["mean_payoff"])
    axes[0].set(title="Alpha demo: mean payoff", xlabel="Variant", ylabel="Payoff")
    axes[0].tick_params(axis="x", rotation=30)
    axes[1].bar(alpha_summary["variant_id"], alpha_summary["mean_entropy"])
    axes[1].set(title="Alpha demo: mean entropy", xlabel="Variant", ylabel="Entropy")
    axes[1].tick_params(axis="x", rotation=30)
    plt.tight_layout()
    plt.show()
    print("Demo readout: stronger gain is a reactivity manipulation; the full paper sweep is in configs/paper/alpha_sweep.toml.")
